# 🤖 Phase 3 — 5 Agents Pipeline

**Project:** AI Resume–Job Description Semantic Matcher with Explainability
**Run after:** `02_model_training.ipynb`

---
## What this notebook does
1. **Cell 1–2** — Install dependencies & imports
2. **Cell 3** — Load all models + config
3. **Cell 4** — Agent 1: Document Parsing (regex + rule-based)
4. **Cell 5** — Agent 2: Skill Normalization (FAISS RAG)
5. **Cell 6** — Agent 3: Semantic Matching (SBERT + XGBoost)
6. **Cell 7** — Agent 4: Gap Analysis (set logic + importance rules)
7. **Cell 8** — Agent 5: Explanation & Recommendations (template-based)
8. **Cell 9** — Full pipeline wired together
9. **Cell 10** — End-to-end test
---
✅ All required files are loaded from `./assets/` folder for easy local execution.

## Cell 1 — Install dependencies

In [15]:
!pip install -q sentence-transformers faiss-cpu xgboost scikit-learn pandas

In [16]:
!pip install -q transformers sentencepiece

## Cell 2 — Imports

In [ ]:
import os, json, pickle, re, ast, warnings
import numpy as np
import pandas as pd
import faiss
import shap
from sentence_transformers import SentenceTransformer
warnings.filterwarnings('ignore')

# Load from assets/ folder for local execution
OUTPUTS_DIR = './assets'
print('✅ Imports done. No API key needed!')

✅ Imports done. No API key needed!


## Cell 3 — Load models & config

In [18]:
from transformers import pipeline

llm_generator = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=-1
)

print("✅ LLM ready")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

✅ LLM ready


In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = """
Candidate has skills: python, machine learning, pandas.
Job requires: python, machine learning, sql, aws.

Write:
Explanation: why candidate scored 75/100.
Improvements: list 3 things candidate should improve.
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("===== FINAL OUTPUT =====\n")
print(result)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


===== FINAL OUTPUT =====

python, machine learning, pandas. Job requires: python, machine learning, sql, aws. Job requires: python, machine learning, sql, aws.


In [20]:
# ── Config ─────────────────────────────────────────────────────────────────
with open(f'{OUTPUTS_DIR}/preprocessing_config.json') as f:
    config = json.load(f)
FEATURE_COLS = config['feature_cols']
print(f'Features: {FEATURE_COLS}')

# ── Models — update filenames to match what you have ──────────────────────
# Check your Colab file panel and update these names if different
with open(f'{OUTPUTS_DIR}/XGB Regressor Model.pkl', 'rb') as f:
    reg_model = pickle.load(f)
with open(f'{OUTPUTS_DIR}/XGB Classifier Model.pkl', 'rb') as f:
    cls_model = pickle.load(f)
with open(f'{OUTPUTS_DIR}/Label Encoder Model.pkl', 'rb') as f:
    le = pickle.load(f)
with open(f'{OUTPUTS_DIR}/Shap Explainer Model Training.pkl', 'rb') as f:
    shap_explainer = pickle.load(f)
print('✅ XGBoost models loaded')

# ── SBERT ──────────────────────────────────────────────────────────────────
print('Loading SBERT... (~1 min first time)')
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ SBERT loaded')

# ── FAISS + skill corpus ───────────────────────────────────────────────────
# Update filenames below if yours are named differently
faiss_index = faiss.read_index(f'{OUTPUTS_DIR}/skill_faiss.index')
with open(f'{OUTPUTS_DIR}/skill_corpus.json') as f:
    skill_corpus = json.load(f)
print(f'✅ FAISS loaded — {len(skill_corpus):,} skills in RAG corpus')

Features: ['feat_skill_overlap', 'feat_resume_skill_count', 'feat_jd_skill_count', 'feat_missing_skill_count', 'feat_edu_match', 'feat_edu_level_resume', 'feat_edu_level_jd', 'feat_exp_years_required', 'feat_resume_text_len', 'feat_sbert_similarity']
✅ XGBoost models loaded
Loading SBERT... (~1 min first time)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SBERT loaded
✅ FAISS loaded — 2,831 skills in RAG corpus


## Cell 4 — Agent 1: Document Parsing Agent
> Rule-based regex parser — no API needed

In [21]:
# ── Known skills dictionary for rule-based extraction ─────────────────────
KNOWN_SKILLS = {
    'python','java','sql','r','scala','c++','javascript','typescript',
    'machine learning','deep learning','nlp','computer vision',
    'tensorflow','pytorch','keras','scikit-learn','xgboost','lightgbm',
    'pandas','numpy','matplotlib','seaborn','plotly',
    'spark','hadoop','kafka','airflow','dbt',
    'aws','azure','gcp','docker','kubernetes','mlflow',
    'mysql','postgresql','mongodb','redis','elasticsearch',
    'git','linux','rest api','flask','fastapi','django',
    'tableau','power bi','excel','statistics','probability',
    'data analysis','data visualization','feature engineering',
    'model deployment','ci/cd','agile','communication'
}

EDU_RANK = {
    'phd':5,'doctorate':5,'ph.d':5,
    'm.tech':4,'m.sc':4,'msc':4,'mba':4,'master':4,'m.e':4,
    'b.tech':3,'b.sc':3,'bsc':3,'b.e':3,'bachelor':3,'be':3,
    'diploma':2,'hsc':1,'ssc':1
}

def extract_skills_from_text(text: str) -> list:
    """Extract skills by matching against known skill dictionary."""
    text_lower = text.lower()
    found = []
    for skill in KNOWN_SKILLS:
        if skill in text_lower:
            found.append(skill)
    # Also extract words after 'skills:' label
    skill_section = re.search(r'skills?[:\s]+([^\n]+)', text_lower)
    if skill_section:
        raw = skill_section.group(1)
        extras = [s.strip() for s in re.split(r'[,|;]', raw) if len(s.strip()) > 2]
        found.extend(extras)
    return list(set(found))

def extract_experience_years(text: str) -> float:
    """Extract years of experience from text."""
    patterns = [
        r'(\d+\.?\d*)\s*\+?\s*years?\s+of\s+experience',
        r'experience[:\s]+(\d+\.?\d*)\s*\+?\s*years?',
        r'(\d+\.?\d*)\s*\+?\s*years?\s+experience',
        r'minimum\s+(\d+)\s+years?',
        r'at\s+least\s+(\d+)\s+years?'
    ]
    for p in patterns:
        m = re.search(p, text.lower())
        if m:
            return float(m.group(1))
    # Count experience entries as proxy
    entries = re.findall(r'(\d+\.?\d*)\s*years?', text.lower())
    if entries:
        return sum(float(e) for e in entries[:3])  # sum first 3 durations
    return 0.0

def extract_education(text: str) -> tuple:
    """Extract highest education level from text."""
    text_lower = text.lower()
    for degree, rank in sorted(EDU_RANK.items(), key=lambda x: -x[1]):
        if degree in text_lower:
            return degree, rank
    return 'bachelor', 3  # default assumption

def agent1_parse_document(resume_text: str, jd_text: str) -> dict:
    """
    Agent 1: Document Parsing Agent (rule-based)
    Extracts structured fields from resume and JD text.
    """
    print('🔄 Agent 1: Parsing documents...')

    # ── Parse Resume ───────────────────────────────────────────────────────
    name_match  = re.search(r'^([A-Z][a-z]+ [A-Z][a-z]+)', resume_text.strip())
    email_match = re.search(r'[\w.+-]+@[\w-]+\.[\w.]+', resume_text)
    resume_skills = extract_skills_from_text(resume_text)
    resume_exp    = extract_experience_years(resume_text)
    resume_degree, resume_edu_rank = extract_education(resume_text)
    resume_len    = len(resume_text.split())

    resume_parsed = {
        'name':                  name_match.group(1) if name_match else 'Candidate',
        'email':                 email_match.group(0) if email_match else None,
        'skills':                resume_skills,
        'total_experience_years': resume_exp,
        'highest_degree':        resume_degree,
        'edu_rank':              resume_edu_rank,
        'text_length':           resume_len
    }

    # ── Parse JD ──────────────────────────────────────────────────────────
    title_match = re.search(r'(?:role|job title|position)[:\s]+([^\n]+)', jd_text, re.I)
    jd_skills   = extract_skills_from_text(jd_text)
    jd_exp      = extract_experience_years(jd_text)
    jd_degree, jd_edu_rank = extract_education(jd_text)

    jd_parsed = {
        'job_title':             title_match.group(1).strip() if title_match else 'Target Role',
        'required_skills':       jd_skills,
        'min_experience_years':  jd_exp,
        'education_requirement': jd_degree,
        'edu_rank':              jd_edu_rank
    }

    print(f'  Resume: {len(resume_skills)} skills, {resume_exp} yrs exp, {resume_degree}')
    print(f'  JD    : {len(jd_skills)} skills required, {jd_exp} yrs min, {jd_degree}')
    print('✅ Agent 1 complete')
    return {'resume': resume_parsed, 'jd': jd_parsed}


# ── Test ───────────────────────────────────────────────────────────────────
sample_resume = """
Priya Sharma | priya@email.com
Skills: Python, Machine Learning, TensorFlow, Pandas, SQL, Git, Data Analysis
Experience:
  - Data Analyst at InfoSys (1.5 years)
  - ML Intern at Wipro (0.5 years)
Education: B.Tech Computer Science, VTU, 2022
"""

sample_jd = """
Role: Data Scientist
Required Skills: Python, Machine Learning, SQL, Docker, AWS, Spark, Deep Learning
Experience Required: Minimum 2 years
Education: B.Tech or higher in CS or related field
"""

parsed = agent1_parse_document(sample_resume, sample_jd)
print()
print('Resume:', json.dumps(parsed['resume'], indent=2))
print('JD    :', json.dumps(parsed['jd'], indent=2))

🔄 Agent 1: Parsing documents...
  Resume: 8 skills, 2.0 yrs exp, b.tech
  JD    : 8 skills required, 2.0 yrs min, b.tech
✅ Agent 1 complete

Resume: {
  "name": "Priya Sharma",
  "email": "priya@email.com",
  "skills": [
    "git",
    "machine learning",
    "pandas",
    "r",
    "data analysis",
    "tensorflow",
    "sql",
    "python"
  ],
  "total_experience_years": 2.0,
  "highest_degree": "b.tech",
  "edu_rank": 3,
  "text_length": 35
}
JD    : {
  "job_title": "Data Scientist",
  "required_skills": [
    "machine learning",
    "r",
    "spark",
    "deep learning",
    "aws",
    "sql",
    "docker",
    "python"
  ],
  "min_experience_years": 2.0,
  "education_requirement": "b.tech",
  "edu_rank": 3
}


## Cell 5 — Agent 2: Skill Normalization Agent (RAG)
> Uses FAISS to map skill synonyms to canonical names

In [22]:
def agent2_normalize_skills(resume_skills: list, jd_skills: list) -> dict:
    """
    Agent 2: Skill Normalization via FAISS RAG.
    Maps raw skill strings to canonical industry names.
    """
    print('🔄 Agent 2: Normalizing skills via RAG...')

    def normalize(skills):
        if not skills:
            return []
        embeddings = sbert.encode(
            [s.lower().strip() for s in skills],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype('float32')
        distances, indices = faiss_index.search(embeddings, k=1)
        normalized = []
        for i, (dist, idx) in enumerate(zip(distances, indices)):
            if float(dist[0]) > 0.75:
                normalized.append(skill_corpus[idx[0]])
            else:
                normalized.append(skills[i].lower().strip())
        return list(set(normalized))

    norm_resume = normalize(resume_skills)
    norm_jd     = normalize(jd_skills)

    print(f'  Resume: {len(resume_skills)} raw → {len(norm_resume)} normalized')
    print(f'  JD    : {len(jd_skills)} raw → {len(norm_jd)} normalized')
    print('✅ Agent 2 complete')
    return {
        'normalized_resume_skills': norm_resume,
        'normalized_jd_skills':     norm_jd
    }


norm_result = agent2_normalize_skills(
    parsed['resume']['skills'],
    parsed['jd']['required_skills']
)
print()
print('Normalized resume skills:', norm_result['normalized_resume_skills'])
print('Normalized JD skills    :', norm_result['normalized_jd_skills'])

🔄 Agent 2: Normalizing skills via RAG...
  Resume: 8 raw → 8 normalized
  JD    : 8 raw → 8 normalized
✅ Agent 2 complete

Normalized resume skills: ['pandas', 'git', 'machine learning', 'r', 'data analysis', 'tensorflow', 'sql', 'python']
Normalized JD skills    : ['machine learning', 'r', 'spark', 'deep learning', 'aws', 'sql', 'docker', 'python']


## Cell 6 — Agent 3: Semantic Matching Agent
> SBERT + XGBoost scoring — no API needed

In [23]:
def agent3_semantic_match(parsed_resume, parsed_jd,
                           norm_resume_skills, norm_jd_skills) -> dict:
    """
    Agent 3: Semantic Matching — computes all scores using
    SBERT embeddings + trained XGBoost model.
    """
    print('🔄 Agent 3: Computing semantic match scores...')

    resume_set   = set(norm_resume_skills)
    jd_set       = set(norm_jd_skills)
    overlap      = resume_set & jd_set
    skill_overlap = len(overlap) / len(jd_set) if jd_set else 0.0

    # SBERT similarity
    emb_r = sbert.encode([' '.join(norm_resume_skills)], normalize_embeddings=True)
    emb_j = sbert.encode([' '.join(norm_jd_skills)],     normalize_embeddings=True)
    sbert_sim = float(np.dot(emb_r, emb_j.T)[0][0])

    # Experience
    resume_exp = float(parsed_resume.get('total_experience_years', 0) or 0)
    jd_exp     = float(parsed_jd.get('min_experience_years', 0) or 0)
    exp_gap    = max(0.0, jd_exp - resume_exp)

    # Education
    edu_resume = parsed_resume.get('edu_rank', 3)
    edu_jd     = parsed_jd.get('edu_rank', 3)
    edu_match  = 1 if edu_resume >= edu_jd else 0

    # Build feature vector matching FEATURE_COLS exactly
    feature_map = {
        'feat_skill_overlap':       skill_overlap,
        'feat_resume_skill_count':  len(norm_resume_skills),
        'feat_jd_skill_count':      len(norm_jd_skills),
        'feat_missing_skill_count': len(jd_set - resume_set),
        'feat_edu_match':           edu_match,
        'feat_edu_level_resume':    edu_resume,
        'feat_edu_level_jd':        edu_jd,
        'feat_exp_years_required':  jd_exp,
        'feat_resume_text_len':     parsed_resume.get('text_length', 100),
        'feat_sbert_similarity':    sbert_sim,
        # fallbacks for any extra features
        'feat_resume_exp_years':    resume_exp,
        'feat_exp_gap':             exp_gap,
    }

    X = pd.DataFrame(
        [[feature_map.get(c, 0) for c in FEATURE_COLS]],
        columns=FEATURE_COLS
    )

    match_score = float(np.clip(reg_model.predict(X)[0], 0, 1))
    label       = le.inverse_transform([cls_model.predict(X)[0]])[0]

    skill_score = round(skill_overlap * 100, 1)
    sbert_score = round(sbert_sim * 100, 1)
    exp_score   = round(max(0, 1 - exp_gap / max(jd_exp, 1)) * 100, 1)
    role_score  = round(match_score * 100, 1)

    print(f'  Overall : {match_score:.3f} → {label}')
    print(f'  Skill   : {skill_score}/100  |  Semantic: {sbert_score}/100  |  Exp: {exp_score}/100')
    print('✅ Agent 3 complete')

    return {
        'match_score':    match_score,
        'overall_score':  round(match_score * 100, 1),
        'label':          label,
        'skill_score':    skill_score,
        'sbert_score':    sbert_score,
        'exp_score':      exp_score,
        'role_score':     role_score,
        'matched_skills': list(overlap),
        'feature_vector': X
    }


match_result = agent3_semantic_match(
    parsed['resume'], parsed['jd'],
    norm_result['normalized_resume_skills'],
    norm_result['normalized_jd_skills']
)
print()
print(f"Score: {match_result['overall_score']}/100  |  Decision: {match_result['label']}")
print(f"Matched skills: {match_result['matched_skills']}")

🔄 Agent 3: Computing semantic match scores...
  Overall : 0.755 → Shortlist
  Skill   : 50.0/100  |  Semantic: 61.6/100  |  Exp: 100.0/100
✅ Agent 3 complete

Score: 75.5/100  |  Decision: Shortlist
Matched skills: ['r', 'python', 'sql', 'machine learning']


## Cell 7 — Agent 4: Gap Analysis Agent
> Rule-based importance classification — no API needed

In [24]:
# Skills that are almost always Must-have for data/ML roles
CRITICAL_SKILLS = {
    'python','sql','machine learning','deep learning','statistics',
    'data analysis','aws','docker','spark','tensorflow','pytorch'
}

def agent4_gap_analysis(norm_resume_skills, norm_jd_skills,
                         parsed_resume, parsed_jd) -> dict:
    """
    Agent 4: Gap Analysis — identifies missing skills
    and classifies them as Must-have or Good-to-have.
    """
    print('🔄 Agent 4: Running gap analysis...')

    resume_set = set(norm_resume_skills)
    jd_set     = set(norm_jd_skills)
    missing    = jd_set - resume_set
    matched    = jd_set & resume_set

    missing_skills = []
    for skill in missing:
        if skill in CRITICAL_SKILLS:
            importance = 'Must-have'
            reason = f'{skill} is a core requirement for this role'
        else:
            importance = 'Good-to-have'
            reason = f'{skill} would strengthen the application'
        missing_skills.append({
            'skill':      skill,
            'importance': importance,
            'reason':     reason
        })

    # Sort: Must-have first
    missing_skills.sort(key=lambda x: 0 if x['importance'] == 'Must-have' else 1)

    must_have    = [s for s in missing_skills if s['importance'] == 'Must-have']
    good_to_have = [s for s in missing_skills if s['importance'] == 'Good-to-have']

    # Experience gap
    resume_exp = float(parsed_resume.get('total_experience_years', 0) or 0)
    jd_exp     = float(parsed_jd.get('min_experience_years', 0) or 0)
    exp_gap    = max(0, jd_exp - resume_exp)

    print(f'  Matched : {len(matched)} skills')
    print(f'  Missing : {len(missing)} ({len(must_have)} must-have, {len(good_to_have)} good-to-have)')
    print(f'  Exp gap : {exp_gap} years')
    print('✅ Agent 4 complete')

    return {
        'missing_skills':      missing_skills,
        'matched_skills':      list(matched),
        'must_have_count':     len(must_have),
        'good_to_have_count':  len(good_to_have),
        'experience_gap':      exp_gap,
        'gap_summary':         f'{len(missing)} gaps: {len(must_have)} critical, {len(good_to_have)} optional'
    }


gap_result = agent4_gap_analysis(
    norm_result['normalized_resume_skills'],
    norm_result['normalized_jd_skills'],
    parsed['resume'], parsed['jd']
)
print()
for s in gap_result['missing_skills']:
    print(f"  [{s['importance']:12}] {s['skill']:25} — {s['reason']}")

🔄 Agent 4: Running gap analysis...
  Matched : 4 skills
  Missing : 4 (4 must-have, 0 good-to-have)
  Exp gap : 0 years
✅ Agent 4 complete

  [Must-have   ] spark                     — spark is a core requirement for this role
  [Must-have   ] docker                    — docker is a core requirement for this role
  [Must-have   ] deep learning             — deep learning is a core requirement for this role
  [Must-have   ] aws                       — aws is a core requirement for this role


## Cell 8 — Agent 5: Explanation & Recommendation Agent


In [25]:
# ===== LLM SETUP (RUN THIS BEFORE AGENT 5) =====

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("✅ LLM ready")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ LLM ready


In [32]:
def agent5_explain_and_recommend(parsed_resume, parsed_jd,
                                  match_result, gap_result) -> dict:
    """
    Agent 5: Explanation & Recommendations + RAW LLM output
    """
    print(' Agent 5: Generating explanation & recommendations...')

    fv    = match_result['feature_vector']
    score = match_result['overall_score']
    label = match_result['label']
    name  = parsed_resume.get('name', 'The candidate')
    role  = parsed_jd.get('job_title', 'the role')

    # ── SHAP explanation ─────────────────────────────────────────────
    shap_vals = shap_explainer.shap_values(fv)

    top_positive = sorted(
        zip(FEATURE_COLS, shap_vals[0]),
        key=lambda x: x[1],
        reverse=True
    )[:3]

    top_negative = sorted(
        zip(FEATURE_COLS, shap_vals[0]),
        key=lambda x: x[1]
    )[:3]

    pos_factors = [
        f.replace('feat_', '').replace('_', ' ')
        for f, v in top_positive if v > 0
    ]

    neg_factors = [
        f.replace('feat_', '').replace('_', ' ')
        for f, v in top_negative if v < 0
    ]

    # ── Base explanation ─────────────────────────────────────────────
    missing_must = [
        s['skill'] for s in gap_result['missing_skills']
        if s['importance'] == 'Must-have'
    ]

    matched = gap_result['matched_skills']
    exp_gap = gap_result['experience_gap']

    if label == 'Shortlist':
        decision_text = f'strong candidate for {role}'
    elif label == 'Maybe':
        decision_text = f'a borderline match for {role}'
    else:
        decision_text = f'not yet ready for {role}'

    explanation = (
        f"{name} scored {score}/100 and is {decision_text}. "
        f"Matching skills include: {', '.join(matched[:5]) if matched else 'none found'}. "
    )

    if missing_must:
        explanation += f"Critical gaps are: {', '.join(missing_must)}. "

    if exp_gap > 0:
        explanation += f"Experience is {exp_gap:.1f} years below requirement. "

    if pos_factors:
        explanation += f"Strengths: {', '.join(pos_factors)}. "

    if neg_factors:
        explanation += f"Areas to improve: {', '.join(neg_factors)}."

    # ── Improvements ────────────────────────────────────────────────
    improvements = []

    if missing_must:
        improvements.append(
            f"Add missing critical skills: {', '.join(missing_must[:3])} with projects."
        )

    if exp_gap > 0:
        improvements.append(
            f"Gain {exp_gap:.1f} more years via internships or projects."
        )

    if match_result['sbert_score'] < 60:
        improvements.append(
            "Use more JD keywords in resume."
        )

    if len(matched) < 3:
        improvements.append(
            "List skills clearly in a Skills section."
        )

    while len(improvements) < 3:
        improvements.append(
            "Improve resume formatting and add measurable achievements."
        )

    # ── LLM GENERATION (RAW — NO FILTERING) ───────────────────────
    try:
        prompt = f"""
Candidate skills: {parsed_resume['skills']}
Job skills: {parsed_jd['required_skills']}

Explain why score is {score}/100 and suggest improvements.
"""

        inputs = llm_tokenizer(prompt, return_tensors="pt")

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7
        )

        llm_text = llm_tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()

    except Exception as e:
        print("LLM error:", e)
        llm_text = "LLM generation failed."

    # ── ATS score ───────────────────────────────────────────────────
    ats_score = min(100, int(
        match_result['skill_score'] * 0.5 +
        match_result['sbert_score'] * 0.3 +
        (100 if gap_result['experience_gap'] == 0 else 50) * 0.2
    ))

    if ats_score >= 75:
        ats_reasoning = 'Well optimized for ATS.'
    elif ats_score >= 50:
        ats_reasoning = 'Partially optimized — add keywords.'
    else:
        ats_reasoning = 'Needs significant ATS improvement.'

    print('✅ Agent 5 complete')

    return {
        'explanation': explanation,
        'llm_explanation': llm_text,   # RAW OUTPUT
        'improvements': improvements[:3],
        'ats_score': ats_score,
        'ats_reasoning': ats_reasoning,
        'pos_factors': pos_factors,
        'neg_factors': neg_factors
    }

## Cell 9 — Full pipeline wired together

In [33]:
def run_full_pipeline(resume_text: str, jd_text: str) -> dict:
    """Master pipeline — runs all 5 agents in sequence."""
    print('=' * 55)
    print('RESUME–JD MATCHER: STARTING FULL PIPELINE')
    print('=' * 55)

    parsed     = agent1_parse_document(resume_text, jd_text)

    normalized = agent2_normalize_skills(
        parsed['resume']['skills'],
        parsed['jd']['required_skills']
    )

    match      = agent3_semantic_match(
        parsed['resume'], parsed['jd'],
        normalized['normalized_resume_skills'],
        normalized['normalized_jd_skills']
    )

    gaps       = agent4_gap_analysis(
        normalized['normalized_resume_skills'],
        normalized['normalized_jd_skills'],
        parsed['resume'], parsed['jd']
    )

    explanation = agent5_explain_and_recommend(
        parsed['resume'], parsed['jd'], match, gaps
    )

    # ✅ FINAL RESULT (UPDATED WITH LLM)
    result = {
        'candidate_name':  parsed['resume'].get('name', 'Unknown'),
        'job_title':       parsed['jd'].get('job_title', 'Unknown'),
        'overall_score':   match['overall_score'],
        'decision':        match['label'],

        'section_scores': {
            'skill_match':  match['skill_score'],
            'semantic_fit': match['sbert_score'],
            'experience':   match['exp_score'],
            'role_fit':     match['role_score']
        },

        'matched_skills':  gaps['matched_skills'],
        'missing_skills':  gaps['missing_skills'],

        # BOTH EXPLANATIONS
        'explanation':        explanation['explanation'],       # SHAP
        'llm_explanation':    explanation['llm_explanation'],   # LLM (NEW)

        'improvements':    explanation['improvements'],
        'ats_score':       explanation['ats_score'],
        'ats_reasoning':   explanation['ats_reasoning']
    }

    print()
    print('=' * 55)
    print('PIPELINE COMPLETE ✅')
    print('=' * 55)

    return result

## Cell 10 — End-to-end test & save output

In [ ]:
test_resume = """
Priya Sharma | priya@email.com
Skills: Python, Machine Learning, TensorFlow, Pandas, SQL, Git, Data Analysis
Experience:
  - Data Analyst at InfoSys (1.5 years)
  - ML Intern at Wipro (0.5 years)
Education: B.Tech Computer Science, VTU, 2022, CGPA 8.4
Projects: Sentiment analysis app, House price prediction model
"""

test_jd = """
Role: Data Scientist
Required Skills: Python, Machine Learning, SQL, Docker, AWS, Spark, Deep Learning
Experience Required: Minimum 2 years
Education: B.Tech in CS or related field
"""

# RUN PIPELINE
result = run_full_pipeline(test_resume, test_jd)

# ── Pretty print ───────────────────────────────────────────────────────────
print()
print('━' * 55)
print(f"  CANDIDATE : {result['candidate_name']}")
print(f"  JOB TITLE : {result['job_title']}")
print(f"  DECISION  : {result['decision']}")
print(f"  SCORE     : {result['overall_score']}/100")
print(f"  ATS SCORE : {result['ats_score']}/100")
print('━' * 55)

print('\nSection Scores:')
for k, v in result['section_scores'].items():
    bar = '█' * int(v // 10) + '░' * (10 - int(v // 10))
    print(f'  {k:15} {bar} {v}/100')

print('\nMatched Skills:', ', '.join(result['matched_skills']) or 'None')

print('\nMissing Skills:')
for s in result['missing_skills']:
    print(f"  [{s['importance']:12}] {s['skill']}")

#  BOTH OUTPUTS
print('\nExplanation (Rule-based):')
print(result['explanation'])

print('\nLLM Explanation (GenAI):')
print(result['llm_explanation'])

print('\nResume Improvement Tips:')
for i, tip in enumerate(result['improvements'], 1):
    print(f'  {i}. {tip}')

print(f"\nATS: {result['ats_score']}/100 — {result['ats_reasoning']}")

# ── Save to local folder (not assets which is read-only) ────────────────────
save = {k: v for k, v in result.items()}

output_file = './sample_pipeline_output.json'
with open(output_file, 'w') as f:
    json.dump(save, f, indent=2, default=str)

print(f'\n✅ Saved: {output_file}')

RESUME–JD MATCHER: STARTING FULL PIPELINE
🔄 Agent 1: Parsing documents...
  Resume: 8 skills, 2.0 yrs exp, b.tech
  JD    : 8 skills required, 2.0 yrs min, b.tech
✅ Agent 1 complete
🔄 Agent 2: Normalizing skills via RAG...
  Resume: 8 raw → 8 normalized
  JD    : 8 raw → 8 normalized
✅ Agent 2 complete
🔄 Agent 3: Computing semantic match scores...
  Overall : 0.755 → Shortlist
  Skill   : 50.0/100  |  Semantic: 61.6/100  |  Exp: 100.0/100
✅ Agent 3 complete
🔄 Agent 4: Running gap analysis...
  Matched : 4 skills
  Missing : 4 (4 must-have, 0 good-to-have)
  Exp gap : 0 years
✅ Agent 4 complete
 Agent 5: Generating explanation & recommendations...
✅ Agent 5 complete

PIPELINE COMPLETE ✅

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CANDIDATE : Priya Sharma
  JOB TITLE : Data Scientist
  DECISION  : Shortlist
  SCORE     : 75.5/100
  ATS SCORE : 63/100
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Section Scores:
  skill_match     █████░░░░░ 50.0/100
  semantic_fi